# adaptive-practice — real Riiid/EdNet run

Runs the full pipeline on the real 100M+ row dataset on a T4 GPU.
Upload your `kaggle.json` when prompted (Kaggle → Settings → API → Create New Token).

In [ ]:
!git clone https://github.com/VijayPraneethSai/adaptive-practice.git
%cd adaptive-practice
!pip install -q -r requirements.txt

In [ ]:
from google.colab import files
import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
uploaded = files.upload()  # pick kaggle.json
shutil.move(next(iter(uploaded)), '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

## 1. Download the data

You must have joined the **Riiid Answer Correctness Prediction** competition on Kaggle (click *Join*/accept rules on its page) before this works.

In [ ]:
!python -m adaptive_practice.data.kaggle_download --out data/riiid_raw
!ls -lh data/riiid_raw

## 2. Build features with Spark

The raw `train.csv`/`questions.csv`/`lectures.csv` already match this project's schema.

In [ ]:
!python -m adaptive_practice.features.spark_job --data data/riiid_raw --out data/riiid_features --memory 10g

## 3. Train retrieval and check recall/latency

In [ ]:
!python -m adaptive_practice.retrieval.two_tower --features data/riiid_features --out runs/riiid --epochs 6 --examples-per-epoch 4000000

## 4. Train the ranker and the gradient-boosting baseline

In [ ]:
!python -m adaptive_practice.ranking.model --features data/riiid_features --out runs/riiid --epochs 6

In [ ]:
!python -m adaptive_practice.ranking.baseline --features data/riiid_features --out runs/riiid --max-train-rows 8000000

## 5. Off-policy and long-term evaluation

This part runs on a quick synthetic simulator (not the real data) because it's the only place a *known-true* answer exists to validate the estimator against — the real log has no such ground truth. It shows the methodology is trustworthy before trusting it on real data, where no ground truth exists to check against.

In [ ]:
!python -m adaptive_practice.evaluation.simulated --run runs/riiid --features data/riiid_features --users 150

## 6. Collect the headline numbers

In [ ]:
import json
for name in ['retrieval_valid', 'ranker_valid', 'baseline_valid', 'ope_validation', 'long_term']:
    path = f'runs/riiid/{name}.json'
    try:
        print(name, '->', json.load(open(path)))
    except FileNotFoundError:
        print(name, 'not found')

## 7. Try it live

In [ ]:
!python -m scripts.demo --run runs/riiid --features data/riiid_features --objective balanced --rounds 6